<a href="https://colab.research.google.com/github/yavuzselimtsx/Lafla-Ai/blob/main/laflagpt_egitim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LaflaGPT - Sifirdan Egitim (Colab / T4)

Bu notebook yeni paket yapisini (`src/laflagpt/`) kullanir. Once depoyu klonlayin, sonra hucreleri sirayla calistirin.
3B egitim T4'te agirdir; once `--smoke` ile boru hattini dogrulayin.

In [ ]:
!pip install torch datasets tokenizers tqdm accelerate safetensors -q
import torch
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print('GPU:', g.name, round(g.total_memory/1e9,1), 'GB')
else:
    print('GPU yok: Runtime > Change runtime type > T4 GPU')

GPU: Tesla T4 15.6 GB


In [3]:
# Depoyu klonla (kendi URL'nizle degistirin)
!git clone https://gitlab.com/lafla2/laflaai.git
%cd laflaai

Cloning into 'laflaai'...
fatal: could not read Username for 'https://gitlab.com': No such device or address
[Errno 2] No such file or directory: 'laflaai'
/content


In [ ]:
import os
os.environ['PYTHONPATH'] = 'src'
# 1) Smoke egitim: boru hattini dogrula
!PYTHONPATH=src python -m laflagpt.egitim.egitim --smoke

In [ ]:
# 2) Thinking SFT verisi -> tokenizer -> 3B egitim
# (egitim_sft.jsonl kendi verinizdir; ornek icin veri/ornek-thinking-sft.jsonl bakiniz)
!PYTHONPATH=src python -m laflagpt.veri.thinking_veri --input veri/ornek-thinking-sft.jsonl --output /content/egitim_sft.jsonl
!PYTHONPATH=src python -m laflagpt.tokenlestirici.tokenizer --input /content/egitim_sft.jsonl --output /content/tok.json --vocab-size 50312

In [ ]:
!PYTHONPATH=src python -m laflagpt.egitim.egitim \
  --model-config konfigurasyon/lafla-3b-model.json \
  --train-config konfigurasyon/egitim-colab-3b.json \
  --data-jsonl /content/egitim_sft.jsonl --tokenizer-path /content/tok.json

In [ ]:
# 3) Sohbet (DEV modu: sansursuz ham thinking)
!PYTHONPATH=src python -m laflagpt.calistir \
  --checkpoint /content/laflagpt-3b-checkpoints/lafla-best \
  --tokenizer /content/tok.json --developer